In [2]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [3]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



PYSPARK_PYTHON: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
# Read raw business data
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [5]:
# Extract the 'categories' column from business data
df_categories = df_read_business_data.select(F.col("categories"))

In [ ]:
# This function is used to get the category name from the categories column
@F.udf(returnType=StringType()) # Define DataFrame schema
def get_category_name(text):

    if text is None:
       return "Not Specified", ["Not Specified"]
    
    list_of_element = text.split(",")
    category_name=list_of_element[0].strip()

    if len(list_of_element) == 0:
        return "Not Specified"
    
    return category_name


df_Category_name = df_categories.withColumn("Category_name", get_category_name(F.col("categories"))).drop(F.col("categories"))

df_distinct_Category_name = df_Category_name.select("Category_name").distinct()

df_surrogate_key = df_distinct_Category_name.withColumn("Category_pk_sk", F.monotonically_increasing_id() + 1)






In [7]:
# Write data to the disk
#df_surrogate_key.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")

In [8]:
#df = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")

In [9]:
#df.filter(F.col("Category_name").isin("Doctors")).count()